In [1]:
import pandas as pd
import duckdb

In [2]:
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

In [3]:
con = duckdb.connect("../data/fpl-mentat.duckdb")

In [ ]:
#con.close()

In [4]:
con.execute("SHOW ALL TABLES").fetchdf()

,database,schema,name,column_names,column_types,temporary
0,fpl-mentat,main,raw_events,"[id, name, deadline_time, release_time, averag...","[BIGINT, VARCHAR, VARCHAR, INTEGER, BIGINT, BO...",False
1,fpl-mentat,main,raw_fixtures,"[code, event, finished, finished_provisional, ...","[BIGINT, BIGINT, BOOLEAN, BOOLEAN, BIGINT, VAR...",False
2,fpl-mentat,main,raw_gw_data,"[id, explain, modified, stats.minutes, stats.g...","[DOUBLE, STRUCT(fixture INTEGER, stats STRUCT(...",False
3,fpl-mentat,main,raw_log,"[id, payload, response_code, table_name, times...","[INTEGER, JSON, VARCHAR, VARCHAR, TIMESTAMP, V...",False
4,fpl-mentat,main,raw_players,"[can_transact, can_select, chance_of_playing_n...","[BOOLEAN, BOOLEAN, DOUBLE, DOUBLE, BIGINT, BIG...",False
5,fpl-mentat,main,raw_teams,"[code, draw, form, id, loss, name, played, poi...","[BIGINT, BIGINT, INTEGER, BIGINT, BIGINT, VARC...",False
6,fpl-mentat,main,stg_deadlines,"[gameweek, deadline_time, ingested_at]","[BIGINT, VARCHAR, TIMESTAMP]",False
7,fpl-mentat,main,stg_fixtures,"[gameweek, fixture_id, kickoff_time, team_a, t...","[BIGINT, BIGINT, VARCHAR, BIGINT, BIGINT, INTE...",False
8,fpl-mentat,main,stg_player_gameweek,"[player_id, gameweek, minutes, goals_scored, a...","[DOUBLE, BIGINT, DOUBLE, DOUBLE, DOUBLE, DOUBL...",False
9,fpl-mentat,main,stg_players,"[player_id, first_name, second_name, known_nam...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, BIGINT, BI...",False


## Double Gameweeks

In [20]:
double = con.execute(
        """
            SELECT COUNT(*) AS games_played
            FROM stg_player_gameweek pg 
            LEFT JOIN stg_players p ON p.player_id = pg.player_id
            LEFT JOIN stg_fixtures f ON f.gameweek = pg.gameweek AND (p.team_id = f.team_a OR p.team_id = f.team_h)
            WHERE pg.gameweek = 26 AND p.player_id = 21
        """
    ).fetchdf()
double.head

,games_played
0,2


In [28]:
double = con.execute(
        """
            SELECT *
            FROM stg_player_gameweek pg 
            LEFT JOIN stg_players p ON p.player_id = pg.player_id
            LEFT JOIN stg_fixtures f ON f.gameweek = pg.gameweek AND (p.team_id = f.team_a OR p.team_id = f.team_h)
            WHERE pg.gameweek = 26 AND p.player_id = 21
        """
    ).fetchdf()
double

,player_id,gameweek,minutes,goals_scored,assists,clean_sheets,goals_conceded,penalties_missed,penalties_saved,yellow_cards,red_cards,saves,bonus,bps,influence,creativity,threat,ict_index,clearances_blocks_interceptions,recoveries,tackles,defensive_contribution,starts,xG,xA,expected_goal_involvements,expected_goals_conceded,in_dreamteam,total_points,ingested_at,player_id_1,first_name,second_name,known_name,team_id,position,penalties_order,corners_and_indirect_freekicks_order,direct_freekicks_order,price,ownership_percent,ingested_at_1,gameweek_1,fixture_id,kickoff_time,team_a,team_h,team_a_score,team_h_score,team_a_difficulty,team_h_difficulty,ingested_at_2
0,21.0,26,180.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,3.0,54.0,56.4,65.6,13.0,13.5,6.0,16.0,6.0,28.0,2.0,0.03,0.69,0.72,1.80,True,14.0,2026-04-16 16:54:01.226083,21,Declan,Rice,,1,3,NaN,1.0,1.0,7.2,24.5,2026-04-16 16:53:52.600819,26,252,2026-02-12T20:00:00Z,1,5,1,1,3,4,2026-04-16 16:54:09.512827
1,21.0,26,180.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,3.0,54.0,56.4,65.6,13.0,13.5,6.0,16.0,6.0,28.0,2.0,0.03,0.69,0.72,1.80,True,14.0,2026-04-16 16:54:01.226083,21,Declan,Rice,,1,3,NaN,1.0,1.0,7.2,24.5,2026-04-16 16:53:52.600819,26,310,2026-02-18T20:00:00Z,1,20,2,2,2,4,2026-04-16 16:54:09.512827


Looks like double gameweek's points are not split up but rather collapsed into one in the player_gameweek table

We can add a flag to count the # of fixtures and set it equal to games_played. This will account for double gameweeks.

In [9]:
teams = con.execute("SELECT * FROM stg_teams").fetchdf()
teams.head

<bound method NDFrame.head of     team_id       team_name                ingested_at
0         1         Arsenal 2026-04-16 16:53:52.580331
1         2     Aston Villa 2026-04-16 16:53:52.580331
2         3         Burnley 2026-04-16 16:53:52.580331
3         4     Bournemouth 2026-04-16 16:53:52.580331
4         5       Brentford 2026-04-16 16:53:52.580331
5         6        Brighton 2026-04-16 16:53:52.580331
6         7         Chelsea 2026-04-16 16:53:52.580331
7         8  Crystal Palace 2026-04-16 16:53:52.580331
8         9         Everton 2026-04-16 16:53:52.580331
9        10          Fulham 2026-04-16 16:53:52.580331
10       11           Leeds 2026-04-16 16:53:52.580331
11       12       Liverpool 2026-04-16 16:53:52.580331
12       13        Man City 2026-04-16 16:53:52.580331
13       14         Man Utd 2026-04-16 16:53:52.580331
14       15       Newcastle 2026-04-16 16:53:52.580331
15       16   Nott'm Forest 2026-04-16 16:53:52.580331
16       17      Sunderland 2026-04

In [10]:
players = con.execute("SELECT * FROM stg_players").fetchdf()
players.head

<bound method NDFrame.head of      player_id first_name            second_name         known_name  team_id  \
0            1      David            Raya Martín                           1   
1            2       Kepa  Arrizabalaga Revuelta                           1   
2            3       Karl                   Hein                           1   
3            4      Tommy                Setford                           1   
4            5    Gabriel   dos Santos Magalhães  Gabriel Magalhães        1   
..         ...        ...                    ...                ...      ...   
821        777     Temple               Ojinnaka                          20   
822        778      Ethan             Sutherland                          20   
823        809     Jerome                  Abbey                          20   
824        816      Angel                  Gomes                          20   
825        817       Adam              Armstrong                          20   

     posi

## Blank Gameweeks

In [23]:
blank = con.execute(
        """
            SELECT *
            FROM stg_player_gameweek pg 
            LEFT JOIN stg_players p ON p.player_id = pg.player_id
            LEFT JOIN stg_fixtures f ON f.gameweek = pg.gameweek AND (p.team_id = f.team_a OR p.team_id = f.team_h)
            WHERE pg.gameweek = 31 AND p.player_id = 21
        """
    ).fetchdf()
blank

,player_id,gameweek,minutes,goals_scored,assists,clean_sheets,goals_conceded,penalties_missed,penalties_saved,yellow_cards,red_cards,saves,bonus,bps,influence,creativity,threat,ict_index,clearances_blocks_interceptions,recoveries,tackles,defensive_contribution,starts,xG,xA,expected_goal_involvements,expected_goals_conceded,in_dreamteam,total_points,ingested_at,player_id_1,first_name,second_name,known_name,team_id,position,penalties_order,corners_and_indirect_freekicks_order,direct_freekicks_order,price,ownership_percent,ingested_at_1,gameweek_1,fixture_id,kickoff_time,team_a,team_h,team_a_score,team_h_score,team_a_difficulty,team_h_difficulty,ingested_at_2
0,21.0,31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,False,0.0,2026-04-16 16:54:01.226083,21,Declan,Rice,,1,3,NaN,1.0,1.0,7.2,24.5,2026-04-16 16:53:52.600819,<NA>,<NA>,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT


In a blank gameweek, the fixtures table won't have a record so the join will return <NA>. This can be used to identify blank gameweeks.

If any of those fields from fixtures is blank, we can make num_fixtures = 0

Due to this, we should probably do a COUNT([field from fixtures]) instead of COUNT(*) since we don't want to count NULLs

In [26]:
blank_and_double = con.execute(
        """
            SELECT 'blank' as gameweek, COUNT(f.fixture_id) AS num_of_fixtures
            FROM stg_player_gameweek pg 
            LEFT JOIN stg_players p ON p.player_id = pg.player_id
            LEFT JOIN stg_fixtures f ON f.gameweek = pg.gameweek AND (p.team_id = f.team_a OR p.team_id = f.team_h)
            WHERE p.player_id = 21 AND pg.gameweek = 31

            UNION

            SELECT 'double' as gameweek, COUNT(f.fixture_id)
            FROM stg_player_gameweek pg 
            LEFT JOIN stg_players p ON p.player_id = pg.player_id
            LEFT JOIN stg_fixtures f ON f.gameweek = pg.gameweek AND (p.team_id = f.team_a OR p.team_id = f.team_h)
            WHERE p.player_id = 21 AND pg.gameweek = 26
        """
    ).fetchdf()
blank_and_double

,gameweek,num_of_fixtures
0,blank,0
1,double,2
